# 04 — Performance Analytics
## Bluestock Mutual Fund Analytics Capstone — D4

**Objective:** Compute, visualise, and export all quantitative performance  
metrics across standard periods (1Y, 3Y, 5Y, Inception).

Metrics Covered
---------------
| Category | Metrics |
|----------|---------|
| Returns | CAGR, Absolute Return |
| Risk | Volatility, Max Drawdown, Skewness, Kurtosis |
| Risk-Adjusted | Sharpe, Sortino, Treynor |
| vs Benchmark | Alpha, Beta, R², Tracking Error, Information Ratio |

**Risk-Free Rate:** 6.5% p.a. (India 10-yr G-Sec proxy)  
**Trading Days:** 252 per year (NOT calendar days)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

BASE    = Path().resolve().parents[1]
DB_PATH = BASE / "datasets" / "db"       / "bluestock_mf.db"
ANA_DIR = BASE / "datasets" / "analytics"
ANA_DIR.mkdir(exist_ok=True)

with sqlite3.connect(DB_PATH) as conn:
    pm = pd.read_sql_query("""
        SELECT pm.*, fm.scheme_name, cm.category, cm.sub_category,
               fhm.amc_name, fm.risk_level, fm.benchmark
        FROM performance_metrics pm
        JOIN fund_master fm ON pm.scheme_code=fm.scheme_code
        JOIN category_master cm ON fm.category_id=cm.category_id
        JOIN fund_house_master fhm ON fm.amc_id=fhm.amc_id
    """, conn)

pm_1y = pm[pm['period_label']=='1Y'].copy()
pm_3y = pm[pm['period_label']=='3Y'].copy()
pm_5y = pm[pm['period_label']=='5Y'].copy()

print(f"Performance metrics loaded: {len(pm)} rows")
print(f"Periods: {pm['period_label'].unique()}")
print(f"Funds:   {pm['scheme_code'].nunique()}")


In [ ]:
# ── 1. RETURNS LEADERBOARD ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, pdata, title in zip(
    axes,
    [pm_1y, pm_3y, pm_5y],
    ['1Y CAGR %', '3Y CAGR %', '5Y CAGR %']
):
    top = pdata.sort_values('cagr', ascending=True).tail(10)
    colors = ['#e74c3c' if v < 0 else '#2ecc71' if v > 0.15 else '#3498db'
              for v in top['cagr']]
    bars = ax.barh([n[:28] for n in top['scheme_name']], top['cagr']*100, color=colors)
    ax.axvline(0, color='black', lw=0.8)
    for bar, val in zip(bars, top['cagr']*100):
        ax.text(val + (0.3 if val >= 0 else -0.3), bar.get_y()+bar.get_height()/2,
                f'{val:.1f}%', va='center', ha='left' if val >= 0 else 'right', fontsize=8)
    ax.set_title(f'Top 10 Funds — {title}', fontweight='bold')
    ax.set_xlabel('CAGR %')

plt.tight_layout()
plt.savefig(ANA_DIR / 'perf_01_cagr_leaderboard.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── 2. FULL METRICS TABLE (1Y) ────────────────────────────────────────────────
display_cols = ['scheme_name','sub_category','risk_level',
                'cagr','volatility_ann','max_drawdown',
                'sharpe_ratio','sortino_ratio','beta','alpha']

table_1y = pm_1y[display_cols].copy()
table_1y.columns = ['Fund','SubCat','Risk',
                    'CAGR%','Vol%','MDD%',
                    'Sharpe','Sortino','Beta','Alpha%']
table_1y['CAGR%']   = (table_1y['CAGR%']   * 100).round(2)
table_1y['Vol%']    = (table_1y['Vol%']    * 100).round(2)
table_1y['MDD%']    = (table_1y['MDD%']    * 100).round(2)
table_1y['Alpha%']  = (table_1y['Alpha%']  * 100).round(2)
table_1y['Sharpe']  = table_1y['Sharpe'].round(3)
table_1y['Sortino'] = table_1y['Sortino'].round(3)
table_1y['Beta']    = table_1y['Beta'].round(3)
table_1y = table_1y.sort_values('CAGR%', ascending=False).reset_index(drop=True)
table_1y['Fund'] = table_1y['Fund'].str[:35]

print("1Y PERFORMANCE METRICS TABLE:")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
print(table_1y.to_string(index=False))
table_1y.to_csv(ANA_DIR / 'performance_1y_table.csv', index=False)


In [ ]:
# ── 3. RISK-RETURN SCATTER (ALL PERIODS) ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
risk_colors = {'Low':'green','Moderate':'limegreen',
               'Moderately High':'orange','High':'darkorange','Very High':'red'}

for ax, pdata, period in zip(axes, [pm_1y, pm_3y, pm_5y], ['1Y','3Y','5Y']):
    for _, row in pdata.iterrows():
        c = risk_colors.get(row['risk_level'],'gray')
        ax.scatter(row['volatility_ann']*100, row['cagr']*100,
                   c=c, s=90, alpha=0.85, edgecolors='white', lw=0.5)
        ax.annotate(row['scheme_name'][:12],
                    (row['volatility_ann']*100, row['cagr']*100),
                    fontsize=6, alpha=0.7,
                    xytext=(3,3), textcoords='offset points')
    ax.axhline(0, color='black', lw=0.7, ls='--')
    ax.set_title(f'{period} Risk vs Return', fontweight='bold')
    ax.set_xlabel('Volatility %')
    ax.set_ylabel('CAGR %')

for label, color in risk_colors.items():
    axes[0].scatter([],[], c=color, s=60, label=label)
axes[0].legend(title='Risk Level', fontsize=7, loc='upper left')

plt.tight_layout()
plt.savefig(ANA_DIR / 'perf_02_risk_return.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── 4. SHARPE & SORTINO COMPARISON ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, metric, title in zip(
    axes,
    ['sharpe_ratio','sortino_ratio'],
    ['Sharpe Ratio (1Y)','Sortino Ratio (1Y)']
):
    data = pm_1y.sort_values(metric, ascending=True)
    colors = ['#e74c3c' if v < 1 else '#f39c12' if v < 2 else '#27ae60'
              for v in data[metric]]
    ax.barh([n[:30] for n in data['scheme_name']], data[metric], color=colors)
    ax.axvline(1, color='black', lw=0.8, ls='--', label='Threshold = 1')
    ax.axvline(2, color='green', lw=0.8, ls='--', label='Good = 2')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Ratio')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(ANA_DIR / 'perf_03_sharpe_sortino.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── 5. ALPHA & BETA ANALYSIS ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Alpha chart
alpha_data = pm_1y.sort_values('alpha', ascending=True)
a_colors = ['#e74c3c' if v < 0 else '#27ae60' for v in alpha_data['alpha']]
axes[0].barh([n[:30] for n in alpha_data['scheme_name']],
             alpha_data['alpha']*100, color=a_colors)
axes[0].axvline(0, color='black', lw=1)
axes[0].set_title("Jensen's Alpha (1Y, %)", fontweight='bold')
axes[0].set_xlabel('Alpha %')

# Beta chart
beta_data = pm_1y.sort_values('beta', ascending=True)
axes[1].barh([n[:30] for n in beta_data['scheme_name']],
             beta_data['beta'], color='steelblue', alpha=0.8)
axes[1].axvline(1.0, color='red', lw=1.2, ls='--', label='Beta = 1 (market)')
axes[1].axvline(0.0, color='black', lw=0.8)
axes[1].set_title('Beta vs Benchmark (1Y)', fontweight='bold')
axes[1].set_xlabel('Beta')
axes[1].legend()

plt.tight_layout()
plt.savefig(ANA_DIR / 'perf_04_alpha_beta.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── 6. DRAWDOWN ANALYSIS ──────────────────────────────────────────────────────
with sqlite3.connect(DB_PATH) as conn:
    nav_df = pd.read_sql_query("""
        SELECT nh.scheme_code, fm.scheme_name, nh.date, nh.nav
        FROM nav_history nh
        JOIN fund_master fm ON nh.scheme_code=fm.scheme_code
        ORDER BY nh.scheme_code, nh.date
    """, conn, parse_dates=['date'])

fig, ax = plt.subplots(figsize=(14, 6))
equity_codes = pm_1y[pm_1y['category']=='Equity']['scheme_code'].values

for code in equity_codes:
    fund = nav_df[nav_df['scheme_code']==code].sort_values('date')
    if len(fund) < 100:
        continue
    peak     = fund['nav'].cummax()
    drawdown = (fund['nav'] - peak) / peak * 100
    ax.plot(fund['date'], drawdown,
            label=fund['scheme_name'].iloc[0][:22], lw=1, alpha=0.8)

ax.fill_between(fund['date'], drawdown, 0, alpha=0.05, color='red')
ax.axhline(0, color='black', lw=0.8)
ax.set_title('Drawdown History — Equity Funds', fontweight='bold')
ax.set_ylabel('Drawdown %')
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(ANA_DIR / 'perf_05_drawdown.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── 7. PERIOD COMPARISON TABLE ────────────────────────────────────────────────
pivot = pm.pivot_table(
    index='scheme_name', columns='period_label',
    values='cagr', aggfunc='first'
) * 100

pivot = pivot[['1Y','3Y','5Y','Inception']].round(2)
pivot.index = [n[:35] for n in pivot.index]

print("CAGR % ACROSS PERIODS:")
print(pivot.to_string())
pivot.to_csv(ANA_DIR / 'performance_all_periods.csv')


In [ ]:
# ── 8. METRICS EXPORT ─────────────────────────────────────────────────────────
export_cols = [
    'scheme_code','scheme_name','sub_category','amc_name','risk_level',
    'period_label','cagr','absolute_return','volatility_ann','max_drawdown',
    'sharpe_ratio','sortino_ratio','treynor_ratio',
    'alpha','beta','r_squared','tracking_error','information_ratio',
    'skewness','kurtosis','risk_free_rate','as_of_date'
]
export = pm[export_cols].copy()
for col in ['cagr','absolute_return','volatility_ann','max_drawdown',
            'alpha','tracking_error']:
    export[col] = (export[col] * 100).round(4)
for col in ['sharpe_ratio','sortino_ratio','treynor_ratio',
            'beta','r_squared','information_ratio','skewness','kurtosis']:
    export[col] = export[col].round(4)

export.to_csv(ANA_DIR / 'performance_metrics_export.csv', index=False)
print(f"Exported: performance_metrics_export.csv  ({len(export)} rows)")
print()
print("FILES WRITTEN:")
for f in sorted(ANA_DIR.glob('perf_*.png')):
    print(f"  {f.name}")
print(f"  performance_1y_table.csv")
print(f"  performance_all_periods.csv")
print(f"  performance_metrics_export.csv")
print()
print("✅ Performance Analytics Complete — proceed to 05_advanced_analytics.ipynb")
